# Link adaptation

> The rate of each transmission (e.g. its MCS): the fastest meeting a target error rate, one for all the copies of a broadcast.

In [ ]:
#| default_exp link_adaptation

In [ ]:
#| hide
from nbdev.showdoc import *

A PHY abstraction with several rates (e.g. the MCSs of a table) rates each transmission at every one of them.
`LinkAdaptation` then picks one per transmission: the fastest whose error rate meets a target, or the slowest if
none does. The copies of a broadcast get one rate, the fastest meeting the target at every receiver: the weakest
receiver's. `SionnaErrorModel` uses it (see `sionna_sys`); it works on numpy arrays and torch tensors.

It is the rule of Sionna SYS's `InnerLoopLinkAdaptation` (`sionna.sys.link_adaptation`), which also has an outer
loop correcting the SINR from acknowledgements (OLLA); one would go here.

In [ ]:
#| export
from __future__ import annotations

from collections import defaultdict
from typing import Sequence

import numpy as np

from comm_core.block import Block, _is_torch

In [ ]:
#| export
class LinkAdaptation(Block):
    "The fastest rate meeting a target error rate (else the slowest): one per transmission, one for all the copies of a broadcast."

    outputs = ('choice',)

    def __init__(
        self,
        target: float = 0.1, # The highest error rate allowed
    ):
        self.target = target

    def call(
        self,
        error_rate, # Error rate of each transmission at each rate, (n, M), the rates from slowest to fastest (numpy or torch)
        group: Sequence | None = None, # The broadcast of each transmission, None for a unicast (None: all unicasts)
    ):
        "Index of the rate chosen for each transmission, (n,)."
        ok = error_rate <= self.target
        if group is not None:                           # a broadcast's copies: a rate meeting the target at all of them
            copies = defaultdict(list)
            for k, g in enumerate(group):
                if g is not None:
                    copies[g].append(k)
            for ks in copies.values():
                ok[ks] = ok[ks].all(0)
        if _is_torch(ok):
            import torch
            fastest = (ok * torch.arange(ok.shape[1], device=ok.device)).argmax(1)
            return torch.where(ok.any(1), fastest, torch.zeros_like(fastest))
        fastest = (ok * np.arange(ok.shape[1])).argmax(1)
        return np.where(ok.any(1), fastest, 0)

In [ ]:
show_doc(LinkAdaptation.call)

In [ ]:
rates = np.array([[0.0, 0.05, 0.5], [0.0, 0.2, 0.9], [0.5, 0.9, 1.0], [0.0, 0.0, 0.01]])
la = LinkAdaptation(target=0.1)
assert la(rates).tolist() == [1, 0, 0, 2]                                 # none meets it (row 2): the slowest
assert la(rates, group=['x', 'x', None, 'y']).tolist() == [0, 0, 0, 2]     # a broadcast: its weakest receiver's

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()